Create a webpage that classifies if the sentence came from the prepared remarks or Q&A, build the naive bayes model and show the naive bayes score



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

In [ ]:
DriveCallFolder = ("/content/drive/MyDrive/Fall 2026/Text Mining/EarningsCallsFolder")

In [ ]:
# collect 200 call links
callURLs = []

for page in range(1, 11):
  if page == 1:
    url = "https://www.fool.com/earnings-call-transcripts/"
  else:
    url = ("https://www.fool.com/earnings-call-transcripts/page/" + str(page) + "/")

  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")

  atags = soup.find_all("a", {"class": "text-gray-1100 group hover:no-underline!"})

  for a in atags:
    callURLs.append("https://www.fool.com" + a["href"])

  print("Page", page, ":", len(atags))

Page 1 : 20
Page 2 : 20
Page 3 : 20
Page 4 : 20
Page 5 : 20
Page 6 : 20
Page 7 : 20
Page 8 : 20
Page 9 : 20
Page 10 : 20


In [ ]:

downloadDir = Path(DriveCallFolder)

df1callRows = []
df2speakerRows = []

# earnings callIDs
for callID, url in enumerate(callURLs, start=1):
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")

  title = soup.find("h1").get_text(" ", strip=True)
  companyName = title.split(" (")[0]

  dateHeading = soup.find(id="date")
  dateText = dateHeading.find_next_sibling("p").get_text(" ", strip=True)


  transcriptHeading = soup.find(id="full-conference-call-transcript")
  paragraphs = transcriptHeading.find_next_siblings("p")

  paragraphTexts = []

  for p in paragraphs:
    paragraphTexts.append(p.get_text(" ", strip=True))

  callText = "\n\n".join(paragraphTexts)

  df1callRows.append({
      "EarningsCallID": callID,
      "CompanyName": companyName,
      "CallDate": dateText,
      "EarningsCallText": callText,
  })

  (downloadDir / f"call_{callID}.txt").write_text(callText, encoding="utf-8")

  section = "Prepared"
  currentTurn = None

  for p in paragraphs:
    text = p.get_text(" ", strip=True)
    bold = p.find("strong")

    #bold is speaker
    if bold and bold.get_text(strip=True).endswith(":"):
      label = bold.get_text(" ", strip=True)
      speakerName = label.rstrip(":").strip()
      speech = text[len(label):].strip()

      if (
        speakerName.lower() == "operator"
        and "first question" in speech.lower()
        ):
        section = "QA"

      currentTurn = {
          "SpeakerTurnID": len(df2speakerRows) + 1,
          "EarningsCallID": callID,
          "SpeakerName": speakerName,
          "PreparedOrQA": section,
          "Text": speech,
      }
      df2speakerRows.append(currentTurn)

    elif currentTurn is not None:
      # Continue the same speaker's turn across paragraphs.
      currentTurn["Text"] += "\n\n" + text

    else:
      raise ValueError(f"Missing initial speaker label: {url}")

  print(callID, companyName, dateText)


1 Santander Brasil Wed, Oct. 28, 2026
2 Ensign Group Wed, Oct. 28, 2026
3 ING Thursday, Oct. 29, 2026
4 MGM Resorts Wednesday, Oct. 28, 2026
5 Boot Barn Oct. 28, 2026
6 Raymond James Wednesday, Oct. 28, 2026
7 Rush Street Interactive Wed, Oct. 28, 2026
8 Q2 Holdings Wednesday, Oct. 28, 2026
9 Gen Digital Wednesday, Oct. 28, 2026
10 Teladoc Wednesday, Oct. 28, 2026
11 Alphabet Wednesday, Oct. 28, 2026
12 Benchmark Electronics Wed, Oct. 28, 2026
13 Carlisle Oct. 28, 2026
14 Fortinet Oct. 28, 2026
15 ServiceNow Wednesday, Oct. 28, 2026
16 Vanda Wednesday, Oct. 28, 2026
17 KLA Wed, Oct. 28, 2026
18 Ford Wed, Oct. 28, 2026
19 Aware Wednesday, Oct. 28, 2026
20 Equinix Oct. 28, 2026
21 Carvana Wed, Oct. 28, 2026
22 PC Connection Wed, Oct. 28, 2026
23 Chipotle Wednesday, Oct. 28, 2026
24 Honeywell Aerospace Wed, Oct 28, 2026
25 eBay Wednesday, Oct. 28, 2026
26 T-Mobile Wed, Oct. 28, 2026
27 Microsoft Wed, Oct. 28, 2026
28 Align Wed, Oct. 28, 2026
29 Sprouts Farmers Market Wednesday, Oct. 28, 2

In [ ]:
# save DFs
df1 = pd.DataFrame(df1callRows)
df2 = pd.DataFrame(df2speakerRows)

df1.to_excel('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrameCE1.xlsx', index=False)
df2.to_excel('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrameCE2.xlsx', index=False)


In [ ]:
df1.head()

,EarningsCallID,CompanyName,CallDate,EarningsCallText
0,1,Santander Brasil,"Wed, Oct. 28, 2026","Camila Toledo: Good morning, everyone, and tha..."
1,2,Ensign Group,"Wed, Oct. 28, 2026","Operator: Hello, everyone. Thank you for joini..."
2,3,ING,"Thursday, Oct. 29, 2026",Operator: Good morning. This is Laura. Welcomi...
3,4,MGM Resorts,"Wednesday, Oct. 28, 2026",Operator: Good afternoon. And welcome to the M...
4,5,Boot Barn,"Oct. 28, 2026","Operator: Good day, everyone. And welcome to t..."


In [ ]:
df2.head()

,SpeakerTurnID,EarningsCallID,SpeakerName,PreparedOrQA,Text
0,1,1,Camila Toledo,Prepared,"Good morning, everyone, and thank you for join..."
1,2,1,Gilson Finkelsztain,Prepared,"Thank you, Camila. Good morning, everyone. I w..."
2,3,1,Carlos Muñoz,Prepared,"Good morning, Gilson, and good morning, everyo..."
3,4,1,Camila Toledo,Prepared,We will now begin the question and answer sess...
4,5,1,Speaker 3,Prepared,Hello and good morning. I want to ask about so...


In [ ]:
!pip install gradio
import pandas as pd
import numpy as np
import gradio as gr
import math

In [ ]:
print(df2.columns)
print()
print(df2["PreparedOrQA"].value_counts())
df2.head()

Index(['SpeakerTurnID', 'EarningsCallID', 'SpeakerName', 'PreparedOrQA',
       'Text'],
      dtype='object')

PreparedOrQA
QA          9608
Prepared    1796
Name: count, dtype: int64


,SpeakerTurnID,EarningsCallID,SpeakerName,PreparedOrQA,Text
0,1,1,Camila Toledo,Prepared,"Good morning, everyone, and thank you for join..."
1,2,1,Gilson Finkelsztain,Prepared,"Thank you, Camila. Good morning, everyone. I w..."
2,3,1,Carlos Muñoz,Prepared,"Good morning, Gilson, and good morning, everyo..."
3,4,1,Camila Toledo,Prepared,We will now begin the question and answer sess...
4,5,1,Speaker 3,Prepared,Hello and good morning. I want to ask about so...


In [ ]:
model_data = df2[["PreparedOrQA", "Text"]].copy()

# Remove missing values
model_data = model_data.dropna()

# Make sure Text is stored as text
model_data["Text"] = model_data["Text"].astype(str)

# Remove blank observations
model_data = model_data[
    model_data["Text"].str.strip() != ""
]

print(model_data["PreparedOrQA"].value_counts())

PreparedOrQA
QA          9605
Prepared    1796
Name: count, dtype: int64


In [ ]:
PreparedText = model_data[
    model_data["PreparedOrQA"] == "Prepared"
]["Text"]

QAText = model_data[
    model_data["PreparedOrQA"] == "QA"
]["Text"]

In [ ]:
PreparedWords = " ".join(PreparedText).lower().split()

QAWords = " ".join(QAText).lower().split()

In [ ]:
PreparedDictionary = {}

for word in PreparedWords:
    if word in PreparedDictionary:
        PreparedDictionary[word] += 1
    else:
        PreparedDictionary[word] = 1

In [ ]:
QADictionary = {}

for word in QAWords:
    if word in QADictionary:
        QADictionary[word] += 1
    else:
        QADictionary[word] = 1

In [ ]:
for word in PreparedDictionary:
    PreparedDictionary[word] = math.log(
        PreparedDictionary[word] / len(PreparedWords)
    )

In [ ]:
for word in QADictionary:
    QADictionary[word] = math.log(
        QADictionary[word] / len(QAWords)
    )

In [ ]:
newSentence = "we expect strong revenue growth"

In [ ]:
newSentenceWords = newSentence.lower().split()

PreparedScore = 0
QAScore = 0

In [ ]:
for i in newSentenceWords:

    if i in PreparedDictionary:

        PreparedScore += PreparedDictionary[i]

    else:

        PreparedScore += math.log(
            1 / len(PreparedWords)
        )

In [ ]:
for i in newSentenceWords:

    if i in QADictionary:

        QAScore += QADictionary[i]

    else:

        QAScore += math.log(
            1 / len(QAWords)
        )

In [ ]:
print("Prepared:", PreparedScore)

print("Q&A:", QAScore)

Prepared: -27.43404010750178
Q&A: -30.715907989415427


In [ ]:
if PreparedScore > QAScore:

    print("Prepared Remarks")

else:

    print("Q&A Response")

Prepared Remarks


In [ ]:
def classify_sentence(sentence):

    # Check if the user entered a sentence
    if sentence is None or sentence.strip() == "":

        return (
            "Please enter a sentence.",
            {},
            "",
            ""
        )

    # Turn the sentence into lowercase words
    newSentenceWords = sentence.lower().split()

    # Start both Naive Bayes scores at zero
    PreparedScore = 0
    QAScore = 0


    # --------------------------------
    # PREPARED REMARKS SCORE
    # --------------------------------

    for i in newSentenceWords:

        if i in PreparedDictionary:

            PreparedScore += PreparedDictionary[i]

        else:

            PreparedScore += math.log(
                1 / len(PreparedWords)
            )


    # --------------------------------
    # Q&A RESPONSE SCORE
    # --------------------------------

    for i in newSentenceWords:

        if i in QADictionary:

            QAScore += QADictionary[i]

        else:

            QAScore += math.log(
                1 / len(QAWords)
            )


    # --------------------------------
    # MODEL DECISION
    # --------------------------------

    if PreparedScore > QAScore:

        prediction = "📜 PREPARED REMARKS"

    else:

        prediction = "❓ Q&A RESPONSE"


    # --------------------------------
    # CONVERT LN SCORES TO PROBABILITY
    # --------------------------------

    # Subtract the largest score first.
    # This prevents very small numbers
    # from causing numerical problems.

    max_score = max(
        PreparedScore,
        QAScore
    )

    PreparedExp = math.exp(
        PreparedScore - max_score
    )

    QAExp = math.exp(
        QAScore - max_score
    )

    total = PreparedExp + QAExp

    PreparedProbability = (
        PreparedExp / total
    )

    QAProbability = (
        QAExp / total
    )


    # --------------------------------
    # PROBABILITY DISPLAY
    # --------------------------------

    probability_output = {
        "📜 Prepared Remarks":
            PreparedProbability,

        "❓ Q&A Response":
            QAProbability
    }


    # --------------------------------
    # LN SCORE DISPLAY
    # --------------------------------

    scores_text = (
        f"Prepared Remarks ln Score: "
        f"{PreparedScore:.4f}\n\n"

        f"Q&A Response ln Score: "
        f"{QAScore:.4f}"
    )


    # --------------------------------
    # EXPLANATION
    # --------------------------------

    explanation = (
        "The classification with the higher "
        "(less negative) ln score is selected "
        "by the Naive Bayes model."
    )


    return (
        prediction,
        probability_output,
        scores_text,
        explanation
    )

In [ ]:
prepared_example = (
    "We delivered strong revenue growth "
    "during the quarter and remain confident "
    "in our long-term strategy."
)

qa_example = (
    "We are very happy with the enthusiasm from "
    "our customers."
)

In [ ]:
total_samples = len(model_data)

prepared_samples = len(PreparedText)

qa_samples = len(QAText)

prepared_vocabulary = len(PreparedDictionary)

qa_vocabulary = len(QADictionary)

total_vocabulary = len(
    set(PreparedWords + QAWords)
)


print("Total Samples:", total_samples)

print(
    "Prepared Samples:",
    prepared_samples
)

print(
    "Q&A Samples:",
    qa_samples
)

print(
    "Total Vocabulary:",
    total_vocabulary
)

Total Samples: 11401
Prepared Samples: 1796
Q&A Samples: 9605
Total Vocabulary: 44894


In [ ]:
custom_css = """

.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
}


/* MAIN TITLE */

#main-title {
    text-align: center;
    padding-top: 20px;
}

#main-title h1 {
    font-size: 48px;
    margin-bottom: 5px;
}

#main-title h3 {
    font-weight: 400;
    opacity: 0.75;
}


/* CLASSIFICATION CARDS */

.class-card {
    border-radius: 22px;
    padding: 30px;
    text-align: center;
    min-height: 220px;
    border: 1px solid rgba(128,128,128,.25);
    box-shadow: 0px 8px 30px rgba(0,0,0,.08);
}

.class-icon {
    font-size: 70px;
    margin-bottom: 10px;
}

.class-card h2 {
    margin: 5px;
}

.class-card p {
    opacity: .75;
}


/* INPUT BOX */

#input-box textarea {
    font-size: 18px !important;
    border-radius: 15px !important;
}


/* ANALYZE BUTTON */

#analyze-button {
    font-size: 18px !important;
    font-weight: bold !important;
}


/* PREDICTION */

#prediction-box textarea {
    font-size: 24px !important;
    font-weight: bold !important;
    text-align: center !important;
}


/* SECTION TITLES */

.section-title {
    margin-top: 25px;
    margin-bottom: 5px;
}


/* STATISTIC BOXES */

.stat-box {
    text-align: center;
    border-radius: 15px;
    padding: 15px;
    border: 1px solid rgba(128,128,128,.25);
}


/* FOOTER */

#footer {
    text-align: center;
    opacity: .65;
    margin-top: 30px;
}

"""

In [ ]:
with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft()
) as demo:


    # =================================
    # HEADER
    # =================================

    gr.Markdown(
        """
        # EARNINGS CALL AI
        ### Can artificial intelligence tell who's speaking?
        """,
        elem_id="main-title"
    )


    gr.Markdown(
        """
        Enter a sentence from an earnings call.

        Our **Naive Bayes classifier** will determine
        whether the sentence is more similar to a
        **Prepared Remark** or a **Q&A Response**.
        """
    )


    # =================================
    # PREPARED VS Q&A CARDS
    # =================================

    with gr.Row():

        with gr.Column():

            gr.HTML(
                """
                <div class="class-card">

                    <div class="class-icon">
                        📜
                    </div>

                    <h2>
                        Prepared Remarks
                    </h2>

                    <p>
                        Language from the scripted
                        portion of an earnings call.
                    </p>

                </div>
                """
            )


        with gr.Column():

            gr.HTML(
                """
                <div class="class-card">

                    <div class="class-icon">
                        ❓
                    </div>

                    <h2>
                        Q&A Response
                    </h2>

                    <p>
                        Language from responses during
                        the question-and-answer section.
                    </p>

                </div>
                """
            )


    # =================================
    # USER INPUT
    # =================================

    gr.Markdown(
        """
        ## 🔍 Test the Model
        """,
        elem_classes="section-title"
    )


    sentence_input = gr.Textbox(
        label=(
            "Enter a New Earnings Call Sentence"
        ),

        placeholder=(
            "Type any earnings call sentence here..."
        ),

        lines=4,

        elem_id="input-box"
    )


    # =================================
    # EXAMPLE BUTTONS
    # =================================

    with gr.Row():

        prepared_button = gr.Button(
            "📜 Try a Prepared Example"
        )

        qa_button = gr.Button(
            "❓ Try a Q&A Example"
        )


    # =================================
    # ANALYZE BUTTON
    # =================================

    analyze_button = gr.Button(
        "ANALYZE SENTENCE →",

        variant="primary",

        elem_id="analyze-button"
    )


    # =================================
    # MODEL DECISION
    # =================================

    gr.Markdown(
        """
        ## 🤖 Model Decision
        """,
        elem_classes="section-title"
    )


    prediction_output = gr.Textbox(
        label="Predicted Classification",

        elem_id="prediction-box"
    )


    # =================================
    # PROBABILITY
    # =================================

    gr.Markdown(
        """
        ## 📊 Classification Probability
        """,
        elem_classes="section-title"
    )


    probability_output = gr.Label(
        label=(
            "Prepared Remarks vs. Q&A Response"
        )
    )


    # =================================
    # LN SCORES
    # =================================

    gr.Markdown(
        """
        ## 🧮 Naive Bayes Classification Scores

        ### Natural Log (ln) Scores
        """,
        elem_classes="section-title"
    )


    score_output = gr.Textbox(
        label="ln Scores",

        lines=5
    )


    explanation_output = gr.Textbox(
        label="How the Classification Works",

        lines=2
    )


    # =================================
    # MODEL STATISTICS
    # =================================

    gr.Markdown(
        """
        ## 🧠 Behind the Model
        """,
        elem_classes="section-title"
    )


    with gr.Row():

        gr.HTML(
            f"""
            <div class="stat-box">

                <h2>
                    {total_samples:,}
                </h2>

                <p>
                    Total Samples
                </p>

            </div>
            """
        )


        gr.HTML(
            f"""
            <div class="stat-box">

                <h2>
                    {prepared_samples:,}
                </h2>

                <p>
                    Prepared Samples
                </p>

            </div>
            """
        )


        gr.HTML(
            f"""
            <div class="stat-box">

                <h2>
                    {qa_samples:,}
                </h2>

                <p>
                    Q&A Samples
                </p>

            </div>
            """
        )


        gr.HTML(
            f"""
            <div class="stat-box">

                <h2>
                    {total_vocabulary:,}
                </h2>

                <p>
                    Vocabulary Words
                </p>

            </div>
            """
        )


    gr.Markdown(
        """
        **Model:** Naive Bayes
        **Classes:** Prepared Remarks and Q&A Responses
        **Scoring:** Word-level natural-log probabilities
        **Classification Rule:** Higher ln score wins
        """
    )


    # =================================
    # FOOTER
    # =================================

    gr.Markdown(
        """
        ---

        **Earnings Call AI**
        • Text Mining Classification Project
        """,
        elem_id="footer"
    )


    # =================================
    # EXAMPLE BUTTON ACTIONS
    # =================================

    prepared_button.click(
        fn=lambda: prepared_example,

        outputs=sentence_input
    )


    qa_button.click(
        fn=lambda: qa_example,

        outputs=sentence_input
    )


    # =================================
    # ANALYZE BUTTON ACTION
    # =================================

    analyze_button.click(

        fn=classify_sentence,

        inputs=sentence_input,

        outputs=[
            prediction_output,
            probability_output,
            score_output,
            explanation_output
        ]
    )

/tmp/ipykernel_1221/3864053745.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 14:19:00 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>